# Olist E-Commerce Analysis - Data Loading & Schema

First thing I need to do is load all the data and get a sense of what I'm working with. There are 9 separate CSV files that all connect to each other somehow - I need to map out the shapes, column names, and where the nulls are before I can do anything useful.

Dataset is from Kaggle (Brazilian E-Commerce Public Dataset by Olist). Covers orders, customers, sellers, payments, reviews and products from 2016–2018.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
# path setup - notebooks live in notebooks/, data is one level up
# os.path.abspath("..") from the notebooks/ dir gives the project root
BASE_DIR = os.path.abspath("..")
RAW_PATH = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_PATH = os.path.join(BASE_DIR, "data", "processed")
CHARTS_PATH = os.path.join(BASE_DIR, "outputs", "charts")

print("data folder:", RAW_PATH)

data folder: /Users/srijitaghosh/Projects/olist-ecommerce-analysis/data/raw


In [3]:
# loading all the tables one by one
customers          = pd.read_csv(os.path.join(RAW_PATH, "olist_customers_dataset.csv"))
orders             = pd.read_csv(os.path.join(RAW_PATH, "olist_orders_dataset.csv"))
order_items        = pd.read_csv(os.path.join(RAW_PATH, "olist_order_items_dataset.csv"))
order_payments     = pd.read_csv(os.path.join(RAW_PATH, "olist_order_payments_dataset.csv"))
order_reviews      = pd.read_csv(os.path.join(RAW_PATH, "olist_order_reviews_dataset.csv"))
products           = pd.read_csv(os.path.join(RAW_PATH, "olist_products_dataset.csv"))
sellers            = pd.read_csv(os.path.join(RAW_PATH, "olist_sellers_dataset.csv"))
geolocation        = pd.read_csv(os.path.join(RAW_PATH, "olist_geolocation_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_PATH, "product_category_name_translation.csv"))

print("all tables loaded - let's go")

all tables loaded - let's go


In [4]:
# checking shape of every table
tables = {
    "orders":               orders,
    "customers":            customers,
    "order_items":          order_items,
    "order_payments":       order_payments,
    "order_reviews":        order_reviews,
    "products":             products,
    "sellers":              sellers,
    "geolocation":          geolocation,
    "category_translation": category_translation
}

for name, df in tables.items():
    print(f"{name} \u2192 {df.shape[0]:,} rows, {df.shape[1]} columns")

orders → 99,441 rows, 8 columns
customers → 99,441 rows, 5 columns
order_items → 112,650 rows, 7 columns
order_payments → 103,886 rows, 5 columns
order_reviews → 99,224 rows, 7 columns
products → 32,951 rows, 9 columns
sellers → 3,095 rows, 4 columns
geolocation → 1,000,163 rows, 5 columns
category_translation → 71 rows, 2 columns


Quick observations:
- orders is the main table with ~99k rows - that's the backbone of everything
- geolocation is huge (~1M rows), I probably won't use all of it
- category_translation is tiny (71 rows) - just a lookup for Portuguese to English names
- order_items has more rows than orders because one order can have multiple products

In [5]:
# just want to see what columns each table has
for name, df in tables.items():
    print(f"\n{name}")
    print(list(df.columns))


orders
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

customers
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

order_items
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

order_payments
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

order_reviews
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

products
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

sellers
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

geolocation
['ge

In [6]:
# checking nulls - only for the 5 tables I'll use most
core_tables = {
    "orders":        orders,
    "order_items":   order_items,
    "order_reviews": order_reviews,
    "products":      products,
    "customers":     customers
}

for name, df in core_tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"{name}: no nulls")
    else:
        print(f"\n{name} - columns with nulls:")
        for col, cnt in nulls.items():
            pct = cnt / len(df) * 100
            print(f"  {col}: {cnt:,} ({pct:.1f}%)")


orders - columns with nulls:
  order_approved_at: 160 (0.2%)
  order_delivered_carrier_date: 1,783 (1.8%)
  order_delivered_customer_date: 2,965 (3.0%)
order_items: no nulls

order_reviews - columns with nulls:
  review_comment_title: 87,656 (88.3%)
  review_comment_message: 58,247 (58.7%)

products - columns with nulls:
  product_category_name: 610 (1.9%)
  product_name_lenght: 610 (1.9%)
  product_description_lenght: 610 (1.9%)
  product_photos_qty: 610 (1.9%)
  product_weight_g: 2 (0.0%)
  product_length_cm: 2 (0.0%)
  product_height_cm: 2 (0.0%)
  product_width_cm: 2 (0.0%)
customers: no nulls


In [7]:
# quick peek at orders - this is the big one
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [8]:
# what order statuses do we have?
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


Looks like most orders are 'delivered' - that's the main status. There are some cancelled and unavailable ones I'll need to filter out. The nulls in the delivery date columns probably belong to those non-delivered orders, which makes sense.

I'll handle all of that in the next notebook.